# Cancer Cell Detection — TensorFlow
### Breast Cancer Wisconsin Dataset | Binary Classification (Malignant vs Benign)

This notebook builds the **same** feedforward neural network as the PyTorch version, but using **TensorFlow/Keras**, to classify tumor samples as malignant or benign from 30 numeric features computed from digitized fine needle aspirate (FNA) biopsy images.

**Dataset:** Breast Cancer Wisconsin (Diagnostic) — built into scikit-learn, 569 samples, 30 features, 2 classes.

**Goal:** Demonstrate a complete, working TensorFlow/Keras training pipeline, directly comparable to the PyTorch notebook — same data, same architecture, same evaluation.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

## 2. Load and Explore the Dataset

Same dataset as the PyTorch notebook: features like mean radius, mean texture, mean perimeter, etc. Target: `0 = malignant`, `1 = benign`.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y
df['diagnosis'] = df['target'].map({0: 'malignant', 1: 'benign'})

print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"\nClass distribution:\n{df['diagnosis'].value_counts()}")
df.head()

In [ ]:
# Quick visualization of class balance and a couple of key features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='diagnosis', ax=axes[0], palette='Set2')
axes[0].set_title('Class Distribution')

sns.scatterplot(data=df, x='mean radius', y='mean texture', hue='diagnosis', ax=axes[1], palette='Set2')
axes[1].set_title('Mean Radius vs Mean Texture')

plt.tight_layout()
plt.show()

## 3. Preprocess the Data

- Train/test split (80/20, stratified)
- Standardize features (zero mean, unit variance)
- Keras can train directly on NumPy arrays — no manual tensor conversion needed.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 4. Define the Model

Same architecture as the PyTorch version:
`Input(30) -> Dense(32, ReLU) -> Dense(16, ReLU) -> Dense(1, Sigmoid)`

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Train the Model

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    verbose=1
)

## 6. Plot Training Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Binary Crossentropy)')
plt.title('Training vs Validation Loss — TensorFlow')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Evaluate on Test Set

In [ ]:
probs = model.predict(X_test, verbose=0)
preds = (probs >= 0.5).astype(int)

acc = accuracy_score(y_test, preds)
print(f"Test Accuracy: {acc:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, preds, target_names=['malignant', 'benign']))

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['malignant', 'benign'], yticklabels=['malignant', 'benign'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — TensorFlow')
plt.show()

## 8. Summary

- Built the same feedforward neural network in **TensorFlow/Keras** for binary classification of breast cancer biopsy data.
- Achieved strong test accuracy with the same 2-hidden-layer architecture as the PyTorch version.
- Pipeline covered: data loading → preprocessing (standardization) → model definition → `.fit()` training → evaluation (accuracy, confusion matrix, classification report).

**Comparison with PyTorch notebook:**
- Both use identical data, splits, and architecture, so results should be very close (small differences are expected due to different default weight initializations and training internals between the two frameworks).
- TensorFlow/Keras required less boilerplate (`.fit()` handles the training loop), while PyTorch gave more explicit control (manual loop) — a useful point to mention if asked about framework tradeoffs.

**Next steps (optional extensions):**
- Try deeper architectures or dropout for regularization
- Use `tf.data.Dataset` for batched/streaming input pipelines
- Try a real cancer cell **image** dataset (e.g. histopathology images) for a CNN-based version